# ClinVar Pathogenic vs Likely Pathogenic — why LP shows a larger average effect

**UKBBGym / Genebass summary-statistics pipeline.**

This notebook reproduces the diagnostic showing that the ClinVar **Likely Pathogenic (LP) > Pathogenic (P)**
average-effect gap is driven by an **allele-count / recurrence confound**, not by a difference in
pathogenicity.

Setup recap (as in the manuscript methods):
- Per-variant effect = Genebass single-variant LMM **beta**, direction-corrected per gene using the
  LOFTEE-burden association sign (all variants in a gene share one external sign — no per-variant
  absolute value).
- Ultra-rare only: carrier count <= 20.
- Category mean = mean direction-corrected z-score across variants in a category per gene-phenotype
  association, aggregated across significant associations (+/- 1.96*SEM across associations).

**Result:** LOFTEE HC (~ singletons) -> LP (~ doubletons) -> P (~ tripletons) is an allele-count ladder
that mirrors, inversely, the effect ladder. Matching both bins on allele count removes the significance
of the LP-P gap.

**Data flow (as in the other analysis notebooks):** the melted per-variant category table is rebuilt
in §1 straight from `MASTER_PATH` (`fetch_hf_data('genebass_annotated.parquet')`) via the same
`load_config` / `scan_variants` / `appv_of` path as `genebass/analysis/mean_phenotype.ipynb` §1 — no
separate local parquet. Parameters go through `env_override`; figures save to `FIG_DIR`. The analysis
in §2–§7 stays **pandas / statsmodels** on purpose: §6's `smf.ols` fixed-effects model and the
pivot/`unstack` pairing are pandas-native, and keeping them untouched keeps every number identical.


## 1. Setup

In [ ]:
# ===================== Setup =====================
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
import statsmodels.formula.api as smf
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

from utils.variant_filtering import (
    load_config, load_variant_class, scan_variants, appv_of, pick_annos,
    derived_schema, env_override, fetch_hf_data,
)

# ===================== Parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override) =====================
# variant_class / categories take their OWN UKBBGYM_PVLP_* names (like mean_phenotype.ipynb's
# UKBBGYM_MEAN_PHENO_*): the shared --variant-class / --selected-categories flags in
# genebass/run_all.sh mean 'missense' there, which would drop the pLoF LOFTEE-HC bin here.
variant_class       = env_override('PVLP_VARIANT_CLASS', 'all_variants')      # membership spans pLoF + any-consequence ClinVar
selected_categories = env_override('PVLP_CATEGORIES', ['loftee', 'clinvar'], 'list')
mac                 = env_override('MAC', 20, int)                           # ultra-rare cap: carrier count <= mac
only_snps           = env_override('ONLY_SNPS', False, bool)
max_variant_length  = env_override('MAX_VARIANT_LENGTH', 50, int)
ci_factor           = env_override('CI_FACTOR', 1.96, float)                 # SEM multiplier for the effect-ladder CIs
AN                  = env_override('AN', 2 * 394841, int)                    # UKB exome participants x2 (diploid); AC_est = AF * AN
ac_thresholds       = [None, 5, 3, 2]                                        # allele-count caps for the matching test

CONFIG_DIR  = str(REPO_ROOT / 'configs')
MASTER_PATH = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
FIG_DIR     = env_override('FIG_DIR', '../../../paper_figures')


In [ ]:
# ===================== Melted per-variant category table =====================
# Rebuilt from MASTER_PATH via the same path as `genebass/analysis/mean_phenotype.ipynb` section 1
# ("Average z-score by category"), so this notebook runs end-to-end from the Hugging Face master
# table with no separate local parquet. Schema of `df`:
#   id                       variant_id (chr:pos:ref:alt)
#   region                   gene_id (Ensembl)
#   phenotype                phenotype name
#   mean_pheno_value_dircor  direction-corrected Genebass beta (per-variant effect)
#   AF                       allele frequency
#   annotation               category flag carried: loftee_hc / clinvar_patho / clinvar_likely_patho / ...
#   annotation_score_dircor  direction-corrected annotation score
#   loftee_corr_dir          sign of the gene-trait association
cfg, all_annos = load_config(CONFIG_DIR, 'config_categories.yaml')
vc  = load_variant_class(CONFIG_DIR, variant_class)
lf  = scan_variants(MASTER_PATH, vc, only_snps=only_snps, max_variant_length=max_variant_length)
annos = pick_annos(cfg, all_annos, selected_categories, derived_schema(MASTER_PATH))

# Binary category flags -> one row per (variant, category it carries). ClinVar regexes are
# non-exclusive by design (Pathogenic also matches Likely_pathogenic), matching avg_zscore_categories.ipynb.
melted = (lf
    .select(set(['id', 'region']) | set(annos))
    .unpivot(index=['id', 'region'], on=annos, variable_name='annotation', value_name='annotation_score')
    .filter(pl.col('annotation_score').cast(pl.Int8) == 1)
    .with_columns(pl.col('annotation_score').cast(pl.Float32), pl.col('region').cast(pl.Utf8))
    .join(cfg.select(['annotation', 'category', 'annotation_dir']).lazy(), on='annotation', how='left')
    .filter(pl.col('category').is_in(selected_categories))
    .unique()
    .with_columns(annotation_score_dircor=pl.when(pl.col('annotation_dir') != 1)
                  .then((pl.col('annotation_score') - 1).abs())
                  .otherwise(pl.col('annotation_score'))))

df = (appv_of(lf, mac)                       # id, region, phenotype, mean_pheno_value, AF  (MAC<=mac proxy)
    .join(melted, on=['id', 'region'], how='inner')
    .join(lf.select(['region', 'phenotype', 'loftee_corr_dir']).unique(),
          on=['region', 'phenotype'], how='inner')
    .with_columns(mean_pheno_value_dircor=pl.col('mean_pheno_value') * pl.col('loftee_corr_dir'))
    .select(['id', 'region', 'phenotype', 'mean_pheno_value_dircor', 'AF',
             'annotation', 'annotation_score_dircor', 'loftee_corr_dir'])
    .collect(engine='streaming')
    .to_pandas())

df['annotation'].value_counts()


## 2. Build the paired category set

We keep the three categories in the figure and restrict to gene-phenotype associations where **all three are present** (Likely Pathogenic is the limiting category -> 203 associations, each a distinct gene).

In [ ]:
CATS = {'loftee_hc', 'clinvar_likely_patho', 'clinvar_patho'}
LABELS = {'loftee_hc': 'LOFTEE HC',
          'clinvar_likely_patho': 'Likely Pathogenic',
          'clinvar_patho': 'Pathogenic'}

sub = df[df['annotation'].isin(CATS)].copy()

# per (gene, phenotype, annotation) mean of direction-corrected effect
g = (sub.groupby(['region', 'phenotype', 'annotation'])['mean_pheno_value_dircor']
        .mean().reset_index())
piv = g.pivot_table(index=['region', 'phenotype'], columns='annotation',
                    values='mean_pheno_value_dircor')

# gene-phenotype pairs where all three categories are present
allthree = piv.dropna(subset=list(CATS))
print(f"paired gene-phenotype associations: {len(allthree)}")
print(f"unique genes: {allthree.reset_index()['region'].nunique()}")

# variant-level table restricted to the paired associations
idx = allthree.index
sub2 = sub.set_index(['region', 'phenotype'])
sub2 = sub2.loc[sub2.index.isin(idx)].reset_index()


## 3. Reproduce the effect ladder and the paired test

In [ ]:
def mean_ci(x):
    x = x.dropna()
    m = x.mean(); h = ci_factor * x.std(ddof=1) / np.sqrt(len(x))
    return m, h, len(x)

for c in ['loftee_hc', 'clinvar_likely_patho', 'clinvar_patho']:
    m, h, n = mean_ci(allthree[c])
    print(f"{LABELS[c]:20s} mean={m:.3f} +/-{h:.3f}  (n={n})")

t, p = stats.ttest_rel(allthree['clinvar_likely_patho'], allthree['clinvar_patho'])
print(f"\npaired t-test  LP vs P: t={t:.3f}, p={p:.4f}")
print(f"mean paired diff (LP-P): {(allthree['clinvar_likely_patho']-allthree['clinvar_patho']).mean():.3f}")

## 4. The allele-count ladder — the mirror image

Estimate allele count from AF using the Genebass exome cohort size (AN ~ 2 x 394,841). **Note:** this is an approximation with a fixed AN — replace with true per-variant AC if available. The ordering and all statistics below that use `AF` directly do **not** depend on this multiplier.

In [ ]:
# AN is set in the Setup parameter block (env_override('AN', 2 * 394841))
sub2['AC_est'] = sub2['AF'] * AN

for c in ['loftee_hc', 'clinvar_likely_patho', 'clinvar_patho']:
    d = sub2.loc[sub2['annotation'] == c, 'AC_est']
    print(f"{LABELS[c]:20s} AC_est: median={d.median():.1f}  mean={d.mean():.1f}  "
          f"frac AC=1: {(d < 1.5).mean():.2f}")


## 5. Gap collapses under allele-count matching

The decisive test: restrict **both** bins to progressively lower allele counts and re-run the paired t-test.

In [ ]:
def paired_gap(max_ac=None):
    s = sub2[sub2['annotation'].isin(['clinvar_patho', 'clinvar_likely_patho'])].copy()
    if max_ac is not None:
        s = s[s['AC_est'] <= max_ac]
    m = (s.groupby(['region', 'phenotype', 'annotation'])['mean_pheno_value_dircor']
           .mean().unstack()
           .dropna(subset=['clinvar_patho', 'clinvar_likely_patho']))
    t, p = stats.ttest_rel(m['clinvar_likely_patho'], m['clinvar_patho'])
    gap = (m['clinvar_likely_patho'] - m['clinvar_patho']).mean()
    return gap, p, len(m)

# max_ac=None means no cap beyond the mac=<MAC> carrier-count filter already applied upstream
# (appv_of), so that row is 'AC<={mac}', not 'all'.
print(f"{'restrict':>10s}  {'LP-P gap':>9s}  {'p':>8s}  {'n_pairs':>7s}")
for mx in ac_thresholds:
    gap, p, n = paired_gap(mx)
    tag = f'AC<={mac}' if mx is None else f'AC<={mx}'
    print(f"{tag:>10s}  {gap:9.3f}  {p:8.4g}  {n:7d}")


## 6. Regression: allele frequency is the dominant within-gene predictor

Variant-level model with gene x phenotype fixed effects (absorbs the pairing). `is_LP` is the residual category effect; `logAF` is the allele-frequency gradient.

In [ ]:
sv = sub2[sub2['annotation'].isin(['clinvar_patho', 'clinvar_likely_patho'])].copy()
sv['logAF'] = np.log10(sv['AF'])
sv['is_LP'] = (sv['annotation'] == 'clinvar_likely_patho').astype(int)
sv['gp'] = sv['region'].str.cat(sv['phenotype'], sep='|')

m1 = smf.ols('mean_pheno_value_dircor ~ is_LP + C(gp)', data=sv).fit()
m2 = smf.ols('mean_pheno_value_dircor ~ is_LP + logAF + C(gp)', data=sv).fit()
print(f"M1 (no AF):   is_LP = {m1.params['is_LP']:.4f}  p={m1.pvalues['is_LP']:.3g}")
print(f"M2 (with AF): is_LP = {m2.params['is_LP']:.4f}  p={m2.pvalues['is_LP']:.3g}"
      f"  | logAF = {m2.params['logAF']:.4f}  p={m2.pvalues['logAF']:.3g}")

# within-gene Spearman of logAF vs effect
tmp = sv.copy()
tmp['eff_dm'] = tmp['mean_pheno_value_dircor'] - tmp.groupby('gp')['mean_pheno_value_dircor'].transform('mean')
tmp['laf_dm'] = tmp['logAF'] - tmp.groupby('gp')['logAF'].transform('mean')
rho, pr = stats.spearmanr(tmp['laf_dm'], tmp['eff_dm'])
print(f"within-gene Spearman(logAF, effect): rho={rho:.3f}, p={pr:.3g}")


## 7. Figures (plotnine, `theme_minimal`)

In [ ]:
pal = {'LOFTEE HC': '#4C72B0', 'Likely Pathogenic': '#DD8452', 'Pathogenic': '#C44E52'}
cat_order = ['Pathogenic', 'Likely Pathogenic', 'LOFTEE HC']  # bottom->top after coord_flip

# Shared theme, identical to the other analysis notebooks (correlations.ipynb, mean_phenotype.ipynb, ...)
_THEME = theme_minimal() + theme(
    axis_text=element_text(size=11, lineheight=1.4),
    axis_title=element_text(size=12),
    legend_text=element_text(size=12),
    legend_title=element_text(size=12),
    plot_background=element_rect(fill='white', color='white'),
)

# This notebook's panels are single-series coord_flip ladders: no legend, subtitle carries the caption.
base_theme = _THEME + theme(legend_position='none', plot_subtitle=element_text(size=12))


### Panel A — effect ladder

In [ ]:
rows = []
for c in ['loftee_hc', 'clinvar_likely_patho', 'clinvar_patho']:
    m, h, n = mean_ci(allthree[c])
    rows.append(dict(label=LABELS[c], mean_pheno=m,
                     ci_low_pheno=m-h, ci_high_pheno=m+h,
                     n_label=f'{n} gene-trait assocs.'))
eff_df = pd.DataFrame(rows)
eff_df['label'] = pd.Categorical(eff_df['label'], categories=cat_order, ordered=True)

pA = (
    ggplot(eff_df, aes(x='label', y='mean_pheno'))
    + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + geom_point(size=3)
    + geom_errorbar(aes(ymin='ci_low_pheno', ymax='ci_high_pheno'), width=0.2)
    + geom_text(aes(label='n_label'), size=11, nudge_x=0.28, ha='center', va='bottom', color='black')
  #   + scale_color_manual(values=pal)
    + labs(
        x='',
        y=f'Mean per-variant phenotype across genes ± {ci_factor} × s.e.m.'
      )
    + coord_flip() 
    + base_theme 
    + theme(
        figure_size=(7, 3*0.6+1.2),
        axis_text=element_text(size=12),
        axis_title=element_text(size=12),
      )
  )
if FIG_DIR:
    pA.save(f'{FIG_DIR}/FS_clinvar_pvlp_effect_ladder.svg', dpi=200, verbose=False)
pA


### Panel B — allele-count ladder (linear scale)

In [ ]:
ac_df = sub2.copy()
ac_df['label'] = ac_df['annotation'].map(LABELS)
ac_df['AC'] = np.clip(ac_df['AC_est'], 0.5, 20)
ac_df['label'] = pd.Categorical(ac_df['label'], categories=cat_order, ordered=True)

pB = (
    ggplot(ac_df, aes(x='label', y='AC'))
    + geom_violin(alpha=0.55, color='none', scale='width', fill='lightgrey')
    + geom_boxplot(width=0.15, outlier_alpha=0, fill='white', color='black')
  #   + scale_fill_manual(values=pal)
    + scale_y_continuous(breaks=[1, 5, 10, 15, 20])
    + labs(
        x='',
        y='Estimated allele count (carriers, UKB exomes)'
      )
    + coord_flip() 
    + base_theme 
    + theme(
        figure_size=(7, 3*0.6+1.2),
        axis_text=element_text(size=12),
        axis_title=element_text(size=12),
      )
  )
if FIG_DIR:
    pB.save(f'{FIG_DIR}/FS_clinvar_pvlp_allele_count_ladder.svg', dpi=200, verbose=False)
pB


### Panel C — gap collapses under allele-count matching

In [ ]:
gap_rows = []
for mx in ac_thresholds:
    baseline = mx is None                        # no AC cap beyond the upstream mac=<MAC> filter
    tag = f'AC ≤ {mac}' if baseline else f'AC ≤ {mx}'
    gap, p, n = paired_gap(mx)
    # '$P$' -> matplotlib mathtext renders just the P italic (its own math font, so no
    # missing-glyph issues); geom_text goes straight to ax.text so the $...$ is parsed.
    gap_rows.append(dict(label=tag, gap=gap, p=p, baseline=baseline,
                         n_label=f'$P$={p:.2g}' + (' **' if p < 0.01 else ' ns')))
gap_df = pd.DataFrame(gap_rows)
gap_order = [r['label'] for r in reversed(gap_rows)]  # smallest cap at the bottom after coord_flip
gap_df['label'] = pd.Categorical(gap_df['label'], categories=gap_order, ordered=True)
gap_fill = {r['label']: ('#9F72BB' if r['baseline'] else '#3DAED4') for r in gap_rows}

pC = (
    ggplot(gap_df, aes(x='label', y='gap', fill='label'))
    + geom_hline(yintercept=0, color='grey')
    + geom_col(alpha=0.85, width=0.6)
    + geom_text(aes(label='n_label'), size=11, nudge_y=0.004, ha='left', va='center', color='black')
    + scale_fill_manual(values=gap_fill)
    + labs(
        x='Restrict both bins to', 
        y='Paired LP − P effect gap'
    )
    + ylim(0, 0.16) 
    + coord_flip() 
    + base_theme 
    + theme(
        figure_size=(7, 4*0.6+1.2),
        axis_text=element_text(size=12),
        axis_title=element_text(size=12),
    )
)
if FIG_DIR:
    pC.save(f'{FIG_DIR}/FS_clinvar_pvlp_gap_collapse.svg', dpi=200, verbose=False)
pC


## 8. Interpretation

- The **LP > P** average-effect gap is real (paired *t* p~0.003) but is driven by
  **allele-count / recurrence composition**, not pathogenicity.
- ClinVar "Pathogenic" preferentially accrues to **recurrent** variants (more reported observations ->
  stronger evidence tier), while the largest-effect ultra-rare variants are held at the lowest allele
  counts by purifying selection. LOFTEE HC, an unbiased population LoF set, is enriched for singletons
  and shows the largest average effect.
- Matching both bins on allele count removes the significance of the gap (p~0.10 at AC<=5, p~0.27 at
  AC<=2), and log(AF) is the dominant within-gene predictor of effect size (p~1e-26).

**Caveat:** allele counts here are estimated as AF x (2 x 394,841). Substitute true per-variant AC for
an exact axis; the conclusion does not depend on the multiplier.